In [28]:
text = """Self-Attention Given a sequence of vectors X ∈ R
n×d where n is the sequence length and d
is the hidden dimension, an attention head first projects X into query, key and value vectors with
WQ,WK,WV ∈ R
d×p where p =
d
h
and h is the number of attention heads:
Q = XWQ K = XWK V = XWV , (1)"""

print(len(text))

289


In [29]:
encoding = text.encode("utf-8")
encoding = list(map(int, encoding))
print(encoding)
print(len(encoding))
original_length = len(encoding)

[83, 101, 108, 102, 45, 65, 116, 116, 101, 110, 116, 105, 111, 110, 32, 71, 105, 118, 101, 110, 32, 97, 32, 115, 101, 113, 117, 101, 110, 99, 101, 32, 111, 102, 32, 118, 101, 99, 116, 111, 114, 115, 32, 88, 32, 226, 136, 136, 32, 82, 10, 110, 195, 151, 100, 32, 119, 104, 101, 114, 101, 32, 110, 32, 105, 115, 32, 116, 104, 101, 32, 115, 101, 113, 117, 101, 110, 99, 101, 32, 108, 101, 110, 103, 116, 104, 32, 97, 110, 100, 32, 100, 10, 105, 115, 32, 116, 104, 101, 32, 104, 105, 100, 100, 101, 110, 32, 100, 105, 109, 101, 110, 115, 105, 111, 110, 44, 32, 97, 110, 32, 97, 116, 116, 101, 110, 116, 105, 111, 110, 32, 104, 101, 97, 100, 32, 102, 105, 114, 115, 116, 32, 112, 114, 111, 106, 101, 99, 116, 115, 32, 88, 32, 105, 110, 116, 111, 32, 113, 117, 101, 114, 121, 44, 32, 107, 101, 121, 32, 97, 110, 100, 32, 118, 97, 108, 117, 101, 32, 118, 101, 99, 116, 111, 114, 115, 32, 119, 105, 116, 104, 10, 87, 81, 44, 87, 75, 44, 87, 86, 32, 226, 136, 136, 32, 82, 10, 100, 195, 151, 112, 32, 119, 104

In [30]:
# There are several issues with this byte pair encoding (BPE) implementation:

# 1. The `byte_pair_map` is not reset in each iteration, so counts accumulate across merges, which is incorrect.
# 2. The merging loop (`for i in range(len(encoding) - 1): ... del encoding[i+1]`) mutates the list while iterating forward, which can skip pairs and cause index errors.
# 3. The code does not handle overlapping merges correctly (should not merge overlapping pairs in a single pass).
# 4. The new token value is not incremented after each merge, so all merges use the same new token.
# 5. The code pops the most frequent pair from the map, but the map is not rebuilt in the next iteration, so it can become inconsistent.
# 6. The code does not check if the most frequent pair occurs more than once (could merge pairs that only occur once, which is not useful).
# 7. The code does not update any mapping from new tokens back to the original pairs, so the merge history is lost.

# A correct BPE implementation would look more like this (pseudocode):

import collections

def bpe(encoding, num_merges=10):
    encoding = list(encoding)
    new_token_value = max(encoding) + 1
    pairs = collections.Counter()
    for i in range(len(encoding) - 1):
        pair = (encoding[i], encoding[i+1])
        pairs[pair] += 1
    
    for count in range(num_merges):
        
        max_pair, freq = pairs.most_common(1)[0]
        if freq < 2:
            break
        print("Most frequent byte pair:", max_pair, freq)
        # Merge all non-overlapping occurrences of max_pair
        i = 0
        new_encoding = []
        while i < len(encoding):
            if i < len(encoding) - 1 and (encoding[i], encoding[i+1]) == max_pair:
                new_encoding.append(new_token_value)
                i += 2
            else:
                new_encoding.append(encoding[i])
                i += 1
        encoding = new_encoding
        new_token_value += 1
        pairs.pop(max_pair)
    return encoding

encoding = bpe(encoding, num_merges=10)



Most frequent byte pair: (101, 110) 9
Most frequent byte pair: (101, 32) 8
Most frequent byte pair: (110, 32) 7
Most frequent byte pair: (104, 101) 7
Most frequent byte pair: (32, 97) 6
Most frequent byte pair: (115, 32) 6
Most frequent byte pair: (32, 88) 5
Most frequent byte pair: (100, 32) 5
Most frequent byte pair: (116, 104) 5
Most frequent byte pair: (110, 116) 4


In [32]:
length_after_bpe = len(encoding)
print(length_after_bpe)
print(original_length/length_after_bpe)

244
1.209016393442623
